# 🦊 Proyecto Pastor Guardián — Detector de Zorros

---

## Descripción
Este notebook implementa el **pipeline completo y estandarizado** para entrenar un detector de zorros usando **YOLO11**, con el objetivo de proteger rebaños de ovejas.

## Estructura del Pipeline

| Celda | Descripción |
|-------|-------------|
| **0** | Configuración global y rutas |
| **1** | Importaciones y verificación de hardware |
| **2** | Auditoría de los datasets originales |
| **3** | Funciones de limpieza y conversión de anotaciones |
| **4** | Consolidación del dataset limpio |
| **5** | Generación del `data.yaml` |
| **6** | Entrenamiento del modelo |
| **7** | Validación y métricas completas |
| **8** | Visualización de resultados |

> ⚠️ **Ejecutar las celdas en orden de arriba a abajo.**

---
## ⚙️ CELDA 0 — Configuración Global
**Modifica solo esta celda** para adaptar el pipeline a tu entorno.
Todas las rutas y parámetros de entrenamiento están centralizados aquí.

In [1]:
# =============================================================
# CONFIGURACIÓN GLOBAL — MODIFICA AQUÍ SI ES NECESARIO
# =============================================================

# Directorio raíz del proyecto
WORKSPACE = r"d:\SOLEDAD"

# Carpeta donde se guardará el dataset consolidado y limpio
OUTPUT_DATASET = WORKSPACE + r"\dataset_zorros_limpio"

# Carpeta donde YOLO guardará los runs de entrenamiento
RUNS_DIR = WORKSPACE + r"\runs"

# Nombre del experimento (aparecerá en la carpeta de resultados)
RUN_NAME = "detector_zorros_final"

# ---- Hiperparámetros de entrenamiento ----
IMGSZ   = 640   # Tamaño de imagen (px). 640 es el estándar de YOLO
EPOCHS  = 100   # Número máximo de épocas
BATCH   = 16    # Imágenes por batch (reducir a 8 si hay error de memoria)
DEVICE  = 0     # 0 = GPU (CUDA), 'cpu' = sin GPU
PATIENCE = 20   # Épocas sin mejora antes de parar (early stopping)

# ---- Rutas de los 3 datasets originales ----
BASE_DS = WORKSPACE + r"\DATOS ANTIGUOS DESCARGADO"
DS1_PATH = BASE_DS + r"\Sorros-yolo.v3i.yolov11"   # Dataset 1: segmentación
DS2_PATH = BASE_DS + r"\Zorros.v2-zorro.yolov11"    # Dataset 2: bbox limpio ✔
DS3_PATH = BASE_DS + r"\fox.v2i.folder"             # Dataset 3: sin labels YOLO

print("✔ Configuración cargada correctamente.")
print(f"  Workspace      : {WORKSPACE}")
print(f"  Dataset destino: {OUTPUT_DATASET}")
print(f"  Runs dir       : {RUNS_DIR}")
print(f"  Épocas máx.    : {EPOCHS}  |  Batch: {BATCH}  |  ImgSz: {IMGSZ}")

✔ Configuración cargada correctamente.
  Workspace      : d:\SOLEDAD
  Dataset destino: d:\SOLEDAD\dataset_zorros_limpio
  Runs dir       : d:\SOLEDAD\runs
  Épocas máx.    : 100  |  Batch: 16  |  ImgSz: 640


---
## 📦 CELDA 1 — Importaciones y Verificación de Hardware
Importamos todas las librerías necesarias y verificamos si hay una GPU disponible para acelerar el entrenamiento.

In [2]:
# ---- Librerías estándar ----
import os
import glob
import shutil
import json
from pathlib import Path

# ---- Librerías científicas ----
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image

# ---- Deep Learning ----
import torch
from ultralytics import YOLO

# ---- Verificación de hardware ----
print("=" * 55)
print("  VERIFICACIÓN DE HARDWARE")
print("=" * 55)

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"  ✔ GPU detectada : {gpu_name}")
    print(f"  ✔ VRAM total    : {gpu_mem:.1f} GB")
    print(f"  ✔ CUDA version  : {torch.version.cuda}")
    device_str = "GPU (CUDA)"
else:
    print("  ⚠ No se detectó GPU. El entrenamiento será en CPU (lento).")
    DEVICE = 'cpu'
    device_str = "CPU"

print(f"  ✔ PyTorch       : {torch.__version__}")
print(f"  ✔ Dispositivo   : {device_str}")
print("=" * 55)

  VERIFICACIÓN DE HARDWARE
  ✔ GPU detectada : NVIDIA GeForce RTX 4060 Laptop GPU
  ✔ VRAM total    : 8.6 GB
  ✔ CUDA version  : 12.1
  ✔ PyTorch       : 2.5.1+cu121
  ✔ Dispositivo   : GPU (CUDA)


---
## 🔍 CELDA 2 — Auditoría de los Datasets Originales
Antes de procesar, analizamos los 3 datasets para entender qué contiene cada uno:
- Cuántas imágenes y etiquetas tiene cada split
- Qué tipo de anotación usan (bbox simple vs. segmentación)

In [3]:
# ---- Función de auditoría ----
def auditar_dataset(nombre, ruta, es_folder=False):
    """
    Analiza un dataset en disco y reporta:
    - Número de imágenes por split
    - Número de etiquetas por split
    - Tipo de anotación detectado (bbox o segmentación)
    """
    print(f"\n  📂 {nombre}")
    print(f"     Ruta: {ruta}")
    total_imgs = 0

    for split in ["train", "valid", "test"]:
        if es_folder:
            # Dataset 3 tiene estructura diferente: split/fox/
            img_dir = os.path.join(ruta, split, "fox")
            lbl_dir = None
        else:
            img_dir = os.path.join(ruta, split, "images")
            lbl_dir = os.path.join(ruta, split, "labels")

        n_imgs = len(glob.glob(os.path.join(img_dir, "*.*"))) if os.path.exists(img_dir) else 0
        n_lbls = len(glob.glob(os.path.join(lbl_dir, "*.txt"))) if (lbl_dir and os.path.exists(lbl_dir)) else 0
        total_imgs += n_imgs

        # Detectar tipo de anotación por tamaño promedio del .txt
        tipo = "Sin labels"
        if lbl_dir and os.path.exists(lbl_dir):
            txts = glob.glob(os.path.join(lbl_dir, "*.txt"))[:20]
            if txts:
                avg = np.mean([os.path.getsize(t) for t in txts])
                tipo = "🔴 Segmentación (polígono)" if avg > 200 else "✅ BBox YOLO"

        if n_imgs > 0:
            print(f"     {split:6s} → {n_imgs:4d} imgs | {n_lbls:4d} labels | {tipo}")

    print(f"     Total imágenes: {total_imgs}")
    return total_imgs

# ---- Ejecutar auditoría ----
print("=" * 60)
print("  AUDITORÍA DE DATASETS ORIGINALES")
print("=" * 60)

t1 = auditar_dataset("DS1 — Sorros-yolo.v3i",    DS1_PATH)
t2 = auditar_dataset("DS2 — Zorros.v2-zorro",     DS2_PATH)
t3 = auditar_dataset("DS3 — fox.v2i.folder",      DS3_PATH, es_folder=True)

print("\n" + "=" * 60)
print(f"  Total imágenes disponibles: {t1 + t2 + t3}")
print("=" * 60)
print("\n  ⚠ DIAGNÓSTICO:")
print("  DS1: Anotaciones en SEGMENTACIÓN → se convertirán a BBox")
print("  DS2: Anotaciones en BBOX YOLO  → uso directo (mejor calidad)")
print("  DS3: Sin labels YOLO válidas   → EXCLUIDO del entrenamiento")

  AUDITORÍA DE DATASETS ORIGINALES

  📂 DS1 — Sorros-yolo.v3i
     Ruta: d:\SOLEDAD\DATOS ANTIGUOS DESCARGADO\Sorros-yolo.v3i.yolov11
     train  →  237 imgs |  237 labels | 🔴 Segmentación (polígono)
     valid  →   12 imgs |   12 labels | 🔴 Segmentación (polígono)
     test   →   12 imgs |   12 labels | 🔴 Segmentación (polígono)
     Total imágenes: 261

  📂 DS2 — Zorros.v2-zorro
     Ruta: d:\SOLEDAD\DATOS ANTIGUOS DESCARGADO\Zorros.v2-zorro.yolov11
     train  →  400 imgs |  400 labels | ✅ BBox YOLO
     valid  →   40 imgs |   40 labels | ✅ BBox YOLO
     test   →   20 imgs |   20 labels | ✅ BBox YOLO
     Total imágenes: 460

  📂 DS3 — fox.v2i.folder
     Ruta: d:\SOLEDAD\DATOS ANTIGUOS DESCARGADO\fox.v2i.folder
     train  →  531 imgs |    0 labels | Sin labels
     valid  →   51 imgs |    0 labels | Sin labels
     test   →   25 imgs |    0 labels | Sin labels
     Total imágenes: 607

  Total imágenes disponibles: 1328

  ⚠ DIAGNÓSTICO:
  DS1: Anotaciones en SEGMENTACIÓN → se co

---
## 🔧 CELDA 3 — Funciones de Conversión y Limpieza
Definimos las funciones que:
1. **Convierten polígonos de segmentación → bounding box** (para el DS1)
2. **Validan** que cada bbox esté dentro de los límites correctos [0-1]
3. **Procesan** cada archivo de etiquetas con seguridad

In [4]:
# ---- Conversión: polígono de segmentación → bounding box ----
def segmentacion_a_bbox(coords):
    """
    Convierte una lista de puntos de polígono [x1,y1,x2,y2,...]
    al formato BBox YOLO: [x_center, y_center, width, height]
    Todos los valores están normalizados entre 0 y 1.
    Retorna None si el bbox no es válido.
    """
    xs = coords[0::2]  # Todos los valores X
    ys = coords[1::2]  # Todos los valores Y

    if not xs or not ys:
        return None

    xmin, xmax = min(xs), max(xs)
    ymin, ymax = min(ys), max(ys)

    x_center = (xmin + xmax) / 2.0
    y_center = (ymin + ymax) / 2.0
    width    = xmax - xmin
    height   = ymax - ymin

    # Validar que el bbox sea razonable (>0 y <=1)
    if width <= 0 or height <= 0 or width > 1.0 or height > 1.0:
        return None
    if not (0 <= x_center <= 1) or not (0 <= y_center <= 1):
        return None

    return x_center, y_center, width, height


# ---- Procesamiento de archivo de etiquetas ----
def procesar_etiqueta(src_path, dst_path, convertir_segmentacion=False):
    """
    Lee un archivo de etiquetas YOLO (.txt) y lo escribe limpio.
    - Si convertir_segmentacion=True, convierte polígonos a bbox.
    - Mapea todas las clases a clase 0 ('zorro').
    - Valida cada línea antes de escribirla.
    Retorna True si se escribieron etiquetas válidas, False si no.
    """
    try:
        with open(src_path, 'r') as f:
            lineas = f.readlines()
    except Exception:
        return False

    lineas_validas = []

    for linea in lineas:
        partes = linea.strip().split()
        if not partes:
            continue  # Línea vacía, ignorar

        try:
            # Extraer coordenadas (ignorar el class_id original → todo es 'zorro')
            coords = [float(x) for x in partes[1:]]

            if len(coords) == 4:
                # Ya es un BBox YOLO: [xc, yc, w, h]
                xc, yc, w, h = coords
                if 0 < w <= 1.0 and 0 < h <= 1.0 and 0 <= xc <= 1 and 0 <= yc <= 1:
                    lineas_validas.append(f"0 {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}\n")

            elif len(coords) > 4 and convertir_segmentacion:
                # Polígono de segmentación → convertir a bbox
                resultado = segmentacion_a_bbox(coords)
                if resultado:
                    xc, yc, w, h = resultado
                    lineas_validas.append(f"0 {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}\n")

        except (ValueError, IndexError):
            continue  # Línea malformada, ignorar

    # Solo escribir si hay al menos una etiqueta válida
    if lineas_validas:
        with open(dst_path, 'w') as f:
            f.writelines(lineas_validas)
        return True

    return False


print("✔ Funciones de conversión y limpieza definidas correctamente.")
print("  - segmentacion_a_bbox(): convierte polígonos → bbox")
print("  - procesar_etiqueta()  : lee, valida y escribe labels limpias")

✔ Funciones de conversión y limpieza definidas correctamente.
  - segmentacion_a_bbox(): convierte polígonos → bbox
  - procesar_etiqueta()  : lee, valida y escribe labels limpias


---
## 📁 CELDA 4 — Consolidación del Dataset Limpio
Unifica los 3 datasets en una única carpeta con estructura YOLO estándar.
- **DS1** → convierte polígonos a bbox válidos
- **DS2** → copia directamente (mejor calidad)
- **DS3** → excluido (no tiene labels YOLO)

In [5]:
# ---- Preparar estructura de carpetas destino ----
if os.path.exists(OUTPUT_DATASET):
    print(f"⚠ Eliminando dataset previo en: {OUTPUT_DATASET}")
    shutil.rmtree(OUTPUT_DATASET)

for split in ["train", "valid", "test"]:
    os.makedirs(os.path.join(OUTPUT_DATASET, split, "images"), exist_ok=True)
    os.makedirs(os.path.join(OUTPUT_DATASET, split, "labels"), exist_ok=True)

print("✔ Estructura de carpetas creada.")

# ---- Función de copia de un dataset ----
def copiar_dataset(nombre_ds, ruta_ds, prefijo, convertir_seg, stats):
    """
    Copia imágenes y etiquetas de un dataset al destino consolidado.
    Aplica conversión de segmentación si es necesario.
    """
    for split in ["train", "valid", "test"]:
        src_imgs = os.path.join(ruta_ds, split, "images")
        src_lbls = os.path.join(ruta_ds, split, "labels")
        dst_imgs = os.path.join(OUTPUT_DATASET, split, "images")
        dst_lbls = os.path.join(OUTPUT_DATASET, split, "labels")

        if not os.path.exists(src_imgs):
            continue  # Este split no existe en el dataset fuente

        imagenes = glob.glob(os.path.join(src_imgs, "*.jpg")) + \
                   glob.glob(os.path.join(src_imgs, "*.jpeg")) + \
                   glob.glob(os.path.join(src_imgs, "*.png"))

        for img_path in imagenes:
            nombre_base = Path(img_path).stem
            lbl_path    = os.path.join(src_lbls, nombre_base + ".txt")

            # Verificar que existe la etiqueta correspondiente
            if not os.path.exists(lbl_path):
                stats[split]["omitidas"] += 1
                continue

            # Nombres de destino con prefijo para evitar colisiones
            nueva_img = os.path.join(dst_imgs, f"{prefijo}_{Path(img_path).name}")
            nueva_lbl = os.path.join(dst_lbls, f"{prefijo}_{nombre_base}.txt")

            # Copiar imagen
            shutil.copy2(img_path, nueva_img)

            # Procesar y copiar etiqueta
            ok = procesar_etiqueta(lbl_path, nueva_lbl, convertir_segmentacion=convertir_seg)

            if ok:
                stats[split]["validas"] += 1
            else:
                # Si la etiqueta no produjo nada válido, eliminar la imagen copiada
                os.remove(nueva_img)
                stats[split]["omitidas"] += 1


# ---- Estadísticas ----
stats = {s: {"validas": 0, "omitidas": 0} for s in ["train", "valid", "test"]}

# ---- Procesar DS1: Sorros-yolo (segmentación → bbox) ----
print("\n▶ Procesando DS1 (Sorros-yolo) — convirtiendo polígonos a bbox...")
copiar_dataset("DS1", DS1_PATH, "ds1", convertir_seg=True, stats=stats)

# ---- Procesar DS2: Zorros.v2-zorro (bbox directo, alta calidad) ----
print("▶ Procesando DS2 (Zorros.v2-zorro) — bbox directo, sin conversión...")
copiar_dataset("DS2", DS2_PATH, "ds2", convertir_seg=False, stats=stats)

# ---- DS3 excluido ----
print("▶ DS3 (fox.v2i.folder) — EXCLUIDO (no contiene labels YOLO).")

# ---- Resumen final ----
print("\n" + "=" * 55)
print("  RESUMEN DE CONSOLIDACIÓN")
print("=" * 55)
total_validas = 0
for split, v in stats.items():
    n = v['validas']
    total_validas += n
    print(f"  {split:6s}: {n:4d} válidas | {v['omitidas']:3d} omitidas")
print(f"  TOTAL : {total_validas} pares imagen-etiqueta listos")
print("=" * 55)

✔ Estructura de carpetas creada.

▶ Procesando DS1 (Sorros-yolo) — convirtiendo polígonos a bbox...
▶ Procesando DS2 (Zorros.v2-zorro) — bbox directo, sin conversión...
▶ DS3 (fox.v2i.folder) — EXCLUIDO (no contiene labels YOLO).

  RESUMEN DE CONSOLIDACIÓN
  train :  637 válidas |   0 omitidas
  valid :   52 válidas |   0 omitidas
  test  :   32 válidas |   0 omitidas
  TOTAL : 721 pares imagen-etiqueta listos


---
## 📄 CELDA 5 — Generación del `data.yaml`
Creamos el archivo de configuración que YOLO necesita para saber dónde están los datos, cuántas clases hay y cómo se llaman.

In [6]:
# ---- Contar imágenes por split para verificación ----
print("=" * 55)
print("  VERIFICACIÓN DEL DATASET CONSOLIDADO")
print("=" * 55)
for split in ["train", "valid", "test"]:
    imgs = glob.glob(os.path.join(OUTPUT_DATASET, split, "images", "*.*"))
    lbls = glob.glob(os.path.join(OUTPUT_DATASET, split, "labels", "*.txt"))
    print(f"  {split:6s}: {len(imgs):4d} imágenes | {len(lbls):4d} etiquetas")
print("=" * 55)

# ---- Crear el data.yaml ----
# Usamos rutas absolutas para evitar problemas de paths relativos
yaml_content = f"""# Configuración del dataset de zorros
# Generado automáticamente por el pipeline

train: {os.path.join(OUTPUT_DATASET, 'train', 'images')}
val:   {os.path.join(OUTPUT_DATASET, 'valid', 'images')}
test:  {os.path.join(OUTPUT_DATASET, 'test',  'images')}

# Número de clases
nc: 1

# Nombre de cada clase
names:
  0: zorro
"""

# Guardar el archivo
DATA_YAML = os.path.join(OUTPUT_DATASET, "data.yaml")
with open(DATA_YAML, 'w') as f:
    f.write(yaml_content)

print(f"\n✔ data.yaml creado en: {DATA_YAML}")
print("\n  Contenido:")
print("-" * 50)
print(yaml_content)
print("-" * 50)

  VERIFICACIÓN DEL DATASET CONSOLIDADO
  train :  637 imágenes |  637 etiquetas
  valid :   52 imágenes |   52 etiquetas
  test  :   32 imágenes |   32 etiquetas

✔ data.yaml creado en: d:\SOLEDAD\dataset_zorros_limpio\data.yaml

  Contenido:
--------------------------------------------------
# Configuración del dataset de zorros
# Generado automáticamente por el pipeline

train: d:\SOLEDAD\dataset_zorros_limpio\train\images
val:   d:\SOLEDAD\dataset_zorros_limpio\valid\images
test:  d:\SOLEDAD\dataset_zorros_limpio\test\images

# Número de clases
nc: 1

# Nombre de cada clase
names:
  0: zorro

--------------------------------------------------


---
## 🚀 CELDA 6 — Entrenamiento del Modelo YOLO11
Entrenamos **YOLO11 Nano** (`yolo11n.pt`) usando el dataset consolidado.
- Augmentación fuerte para mejorar generalización
- Optimizador AdamW con learning rate scheduler
- Early stopping para evitar sobreajuste

In [7]:
# ---- Cargar modelo base preentrenado ----
# yolo11n.pt está preentrenado en COCO (80 clases)
# YOLO automáticamente adapta la cabeza de detección a nc=1 (zorro)
print("▶ Cargando modelo YOLO11 Nano preentrenado...")
model = YOLO("yolo11n.pt")

print(f"▶ Iniciando entrenamiento con {EPOCHS} épocas máximas...")
print(f"  Datos        : {DATA_YAML}")
print(f"  Resultados   : {os.path.join(RUNS_DIR, RUN_NAME)}")
print()

# ---- Entrenamiento ----
train_results = model.train(
    data     = DATA_YAML,      # Archivo de configuración del dataset
    epochs   = EPOCHS,         # Épocas máximas de entrenamiento
    imgsz    = IMGSZ,          # Tamaño de imagen (640x640)
    batch    = BATCH,          # Imágenes por batch
    device   = DEVICE,         # GPU o CPU
    project  = RUNS_DIR,       # Carpeta de resultados
    name     = RUN_NAME,       # Nombre del experimento
    exist_ok = True,           # Sobrescribir si ya existe el run
    patience = PATIENCE,       # Early stopping: parar si no mejora

    # ---- Augmentación de datos ----
    # Ayuda al modelo a generalizar mejor en condiciones variadas
    hsv_h    = 0.015,          # Variación de tono (color)
    hsv_s    = 0.7,            # Variación de saturación
    hsv_v    = 0.4,            # Variación de brillo
    degrees  = 10.0,           # Rotación aleatoria ±10°
    translate= 0.1,            # Traslación aleatoria 10%
    scale    = 0.5,            # Escala aleatoria ±50%
    flipud   = 0.5,            # Volteo vertical con 50% de probabilidad
    fliplr   = 0.5,            # Volteo horizontal con 50% de probabilidad
    mosaic   = 1.0,            # Mosaic: combina 4 imágenes en 1 (activo siempre)
    mixup    = 0.1,            # MixUp: mezcla 2 imágenes con 10% de prob.

    # ---- Regularización y optimizador ----
    optimizer    = "AdamW",    # AdamW: mejor convergencia que SGD en datasets pequeños
    lr0          = 0.001,      # Learning rate inicial
    lrf          = 0.01,       # Factor LR final (lr_final = lr0 * lrf)
    weight_decay = 0.0005,     # L2 regularización para evitar sobreajuste
    warmup_epochs= 3.0,        # Épocas de calentamiento del LR

    # ---- Configuración de guardado ----
    save        = True,        # Guardar checkpoints
    save_period = 10,          # Guardar cada 10 épocas
    plots       = True,        # Generar gráficos de entrenamiento
    val         = True,        # Validar en cada época
    verbose     = True,        # Mostrar progreso detallado
)

# ---- Ruta del mejor modelo guardado ----
RUN_DIR    = os.path.join(RUNS_DIR, RUN_NAME)
BEST_MODEL = os.path.join(RUN_DIR, "weights", "best.pt")

print(f"\n✔ Entrenamiento completado.")
print(f"  Mejor modelo guardado en: {BEST_MODEL}")

▶ Cargando modelo YOLO11 Nano preentrenado...
▶ Iniciando entrenamiento con 100 épocas máximas...
  Datos        : d:\SOLEDAD\dataset_zorros_limpio\data.yaml
  Resultados   : d:\SOLEDAD\runs\detector_zorros_final

Ultralytics 8.4.60  Python-3.12.10 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=d:\SOLEDAD\dataset_zorros_limpio\data.yaml, degrees=10.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_widt

---
## 📊 CELDA 7 — Validación y Métricas Completas
Evaluamos el modelo entrenado sobre el conjunto de **test** y calculamos:
- **mAP@50** y **mAP@50-95** (métricas principales de detección)
- **Precisión** (qué % de detecciones son correctas)
- **Recall / Sensibilidad** (qué % de zorros reales detecta)
- **F1-Score** (balance entre precisión y recall)

In [8]:
# ---- Cargar el mejor modelo entrenado ----
if not os.path.exists(BEST_MODEL):
    print(f"✘ ERROR: No se encontró el modelo en {BEST_MODEL}")
    print("  Asegúrate de haber ejecutado la Celda 6 primero.")
else:
    print(f"▶ Cargando mejor modelo: {BEST_MODEL}")
    best_model = YOLO(BEST_MODEL)

    # ---- Validación sobre el conjunto de TEST ----
    print("▶ Ejecutando validación sobre el conjunto de test...\n")
    metrics = best_model.val(split="test", plots=True)

    # ---- Extraer métricas principales ----
    mAP50    = metrics.box.map50
    mAP5095  = metrics.box.map
    precision = metrics.box.mp
    recall    = metrics.box.mr

    # F1-Score = media armónica de precisión y recall
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0.0

    # ---- Reporte de métricas ----
    print()
    print("  " + "═" * 56)
    print("  ║    RESULTADOS FINALES — DETECTOR DE ZORROS          ║")
    print("  " + "═" * 56)
    print(f"  ║  mAP@50        : {mAP50:.4f}   ({mAP50*100:.2f}%)              ║")
    print(f"  ║  mAP@50-95     : {mAP5095:.4f}   ({mAP5095*100:.2f}%)              ║")
    print(f"  ║  Precisión     : {precision:.4f}   ({precision*100:.2f}%)              ║")
    print(f"  ║  Recall        : {recall:.4f}   ({recall*100:.2f}%)              ║")
    print(f"  ║  F1-Score      : {f1:.4f}   ({f1*100:.2f}%)              ║")
    print("  " + "═" * 56)

    # ---- Métricas por clase ----
    print("\n  Desglose por clase:")
    print("  " + "-" * 60)
    nombres_clases = best_model.names
    clases_eval = getattr(metrics, 'classes', list(range(len(metrics.box.p))))
    for i, c in enumerate(clases_eval):
        clase = nombres_clases.get(int(c), f"Clase_{c}")
        p    = metrics.box.p[i]  if i < len(metrics.box.p)  else 0.0
        r    = metrics.box.r[i]  if i < len(metrics.box.r)  else 0.0
        ap50 = metrics.box.ap50[i] if i < len(metrics.box.ap50) else 0.0
        ap   = metrics.box.ap[i]   if i < len(metrics.box.ap)   else 0.0
        fi   = (2*p*r/(p+r)) if (p+r) > 0 else 0.0
        print(f"  Clase '{clase}':")
        print(f"    Precisión={p:.4f} | Recall={r:.4f} | F1={fi:.4f} | AP50={ap50:.4f} | AP50-95={ap:.4f}")
    print("  " + "-" * 60)

    # ---- Guardar métricas en JSON ----
    reporte = {
        "modelo": BEST_MODEL,
        "mAP50":     round(float(mAP50), 4),
        "mAP50_95":  round(float(mAP5095), 4),
        "precision": round(float(precision), 4),
        "recall":    round(float(recall), 4),
        "f1_score":  round(float(f1), 4),
    }
    reporte_path = os.path.join(RUN_DIR, "reporte_metricas.json")
    with open(reporte_path, 'w') as f:
        json.dump(reporte, f, indent=4)
    print(f"\n✔ Reporte JSON guardado en: {reporte_path}")

▶ Cargando mejor modelo: d:\SOLEDAD\runs\detector_zorros_final\weights\best.pt
▶ Ejecutando validación sobre el conjunto de test...

Ultralytics 8.4.60  Python-3.12.10 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
YOLO11n summary (fused): 101 layers, 2,582,347 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 3.40.9 MB/s, size: 43.2 KB)
val: Scanning D:\SOLEDAD\dataset_zorros_limpio\test\labels... 32 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 32/32 407.7it/s 0.1s
val: New cache created: D:\SOLEDAD\dataset_zorros_limpio\test\labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 2/2 4.9s/it 9.8s<30.2s
                   all         32         34      0.964      0.971      0.986      0.728
Speed: 13.5ms preprocess, 21.0ms inference, 0.0ms loss, 3.4ms postprocess per image
Results saved to D:\SOLEDAD\runs\detect\val-3

  ════════════════════════════════════

---
## 📈 CELDA 8 — Visualización de Resultados
Mostramos todos los gráficos generados automáticamente por YOLO durante el entrenamiento y la validación:
- Curvas de pérdida y mAP por época
- Matriz de confusión
- Curvas Precisión-Recall y F1

In [9]:
# ---- Lista de gráficos a mostrar ----
# Cada entrada: (título descriptivo, nombre del archivo)
graficos = [
    ("Diagrama de Aprendizaje (Pérdidas y mAP por época)", "results.png"),
    ("Matriz de Confusión",                               "confusion_matrix.png"),
    ("Matriz de Confusión Normalizada",                   "confusion_matrix_normalized.png"),
    ("Curva Precisión-Recall (PR)",                       "PR_curve.png"),
    ("Curva F1 vs. Confianza",                            "F1_curve.png"),
    ("Curva Precisión vs. Confianza",                     "P_curve.png"),
    ("Curva Recall vs. Confianza",                        "R_curve.png"),
]

# ---- Mostrar cada gráfico ----
for titulo, archivo in graficos:
    # Buscar en el directorio del run actual
    ruta = os.path.join(RUN_DIR, archivo)
    if os.path.exists(ruta):
        img = Image.open(ruta)
        plt.figure(figsize=(12, 8))
        plt.imshow(img)
        plt.axis('off')
        plt.title(titulo, fontsize=14, fontweight='bold', pad=12)
        plt.tight_layout()
        plt.show()
    else:
        print(f"⚠ No encontrado: {archivo}")

<Figure size 1200x800 with 1 Axes>

<Figure size 1200x800 with 1 Axes>

<Figure size 1200x800 with 1 Axes>

⚠ No encontrado: PR_curve.png
⚠ No encontrado: F1_curve.png
⚠ No encontrado: P_curve.png
⚠ No encontrado: R_curve.png


---
## 📊 CELDA 9 — Gráfico Resumen de Métricas
Gráfico de barras horizontal con todas las métricas, con un código de colores para interpretar el rendimiento de un vistazo.

In [10]:
# ---- Datos para el gráfico resumen ----
nombres_metricas = ["mAP@50", "mAP@50-95", "Precisión", "Recall", "F1-Score"]
valores_metricas = [
    reporte["mAP50"],
    reporte["mAP50_95"],
    reporte["precision"],
    reporte["recall"],
    reporte["f1_score"],
]

# ---- Código de colores según rendimiento ----
# Verde ≥ 70% | Naranja 50-70% | Rojo < 50%
colores = [
    "#4CAF50" if v >= 0.7 else "#FF9800" if v >= 0.5 else "#F44336"
    for v in valores_metricas
]

# ---- Crear figura ----
fig, ax = plt.subplots(figsize=(11, 5))
bars = ax.barh(nombres_metricas, valores_metricas, color=colores, height=0.55, edgecolor='white')

# Líneas de referencia
ax.axvline(0.7,  color='#4CAF50', linestyle='--', alpha=0.6, linewidth=1.5, label='Bueno ≥ 70%')
ax.axvline(0.5,  color='#FF9800', linestyle='--', alpha=0.6, linewidth=1.5, label='Aceptable ≥ 50%')

# Etiquetas de valor en cada barra
for bar, val in zip(bars, valores_metricas):
    ax.text(
        val + 0.01,
        bar.get_y() + bar.get_height() / 2,
        f"{val:.4f}  ({val*100:.1f}%)",
        va='center', fontsize=11, fontweight='bold', color='#222'
    )

# Leyenda de colores
leyenda = [
    mpatches.Patch(color='#4CAF50', label='≥ 70%  Bueno'),
    mpatches.Patch(color='#FF9800', label='50-70% Aceptable'),
    mpatches.Patch(color='#F44336', label='< 50%  Bajo'),
]
ax.legend(handles=leyenda, loc='lower right', fontsize=10)

ax.set_xlim(0, 1.15)
ax.set_xlabel("Valor (0 = peor → 1 = mejor)", fontsize=11)
ax.set_title("📊 Resumen de Métricas — Detector de Zorros", fontsize=14, fontweight='bold', pad=14)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()

# Guardar el gráfico
ruta_resumen = os.path.join(RUN_DIR, "resumen_metricas.png")
plt.savefig(ruta_resumen, dpi=150, bbox_inches='tight')
plt.show()

print(f"✔ Gráfico resumen guardado en: {ruta_resumen}")
print()
print("=" * 50)
print("  PIPELINE COMPLETADO EXITOSAMENTE")
print("=" * 50)
print(f"  Mejor modelo : {BEST_MODEL}")
print(f"  mAP@50       : {reporte['mAP50']*100:.2f}%")
print(f"  F1-Score     : {reporte['f1_score']*100:.2f}%")
print("=" * 50)

C:\Users\NITRO\AppData\Local\Temp\ipykernel_18420\3656617863.py:48: UserWarning: Glyph 128202 (\N{BAR CHART}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\NITRO\AppData\Local\Temp\ipykernel_18420\3656617863.py:52: UserWarning: Glyph 128202 (\N{BAR CHART}) missing from font(s) DejaVu Sans.
  plt.savefig(ruta_resumen, dpi=150, bbox_inches='tight')


<Figure size 1100x500 with 1 Axes>

✔ Gráfico resumen guardado en: d:\SOLEDAD\runs\detector_zorros_final\resumen_metricas.png

  PIPELINE COMPLETADO EXITOSAMENTE
  Mejor modelo : d:\SOLEDAD\runs\detector_zorros_final\weights\best.pt
  mAP@50       : 98.60%
  F1-Score     : 96.73%
